# Notebook 1 · Lexical decision data with pandas

The `lexdec` dataset contains 1,659 lexical-decision trials from 21 participants and 79
English nouns. Participants decided whether each stimulus was a word. The variables
include response accuracy, log reaction time, trial number, native-language group, word
frequency, word length, and semantic class.

Source: `languageR`, Baayen (2008).

In [ ]:
from pathlib import Path

class Check:
    def _result(self, passed, success, hint):
        if passed:
            print(f"✅ {success}")
        else:
            print("✗ Not correct. Open the hint if needed.")
        return passed

    def equal(self, actual, expected, success="Correct.", hint="Value does not match the expected result."):
        try:
            passed = actual == expected
            if hasattr(passed, "all"):
                passed = bool(passed.all())
        except Exception:
            passed = False
        return self._result(bool(passed), success, hint)

    def shape(self, actual, expected, success="Shape is correct.", hint="Shape is incorrect."):
        return self._result(tuple(actual.shape) == tuple(expected), success, hint)

    def columns(self, frame, expected, success="Columns are correct.", hint="Inspect frame.columns and select with a list of names."):
        return self._result(list(frame.columns) == list(expected), success, hint)

    def choice(self, actual, expected, explanations):
        normalised = str(actual).strip().upper()
        hint = explanations.get(normalised, "Choose one of the listed letters.")
        return self._result(normalised == expected.upper(), "Correct.", hint)

check = Check()

## 1 · Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates = [
    Path("book/data/real/lexical_decision.csv"),
    Path("../data/real/lexical_decision.csv"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Open this notebook from the workshop repository.")

trials = ...  # load the CSV
trials.head()

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Use `pd.read_csv(data_path)`.

</details>

In [ ]:
check.shape(trials, (1659, 28), hint="Read the CSV at data_path with pandas.")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
trials = pd.read_csv(data_path)
trials.head()
```

</details>

## 2 · Inspect the DataFrame

Inspect `.shape`, `.columns`, and `.dtypes`. Assign the number of trials, participants,
and words to the three variables below.

In [ ]:
print("shape:", trials.shape)
print("columns:", trials.columns.tolist())
print("types:\n", trials.dtypes)

n_trials = ...
n_participants = ...
n_words = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Use `len(trials)` for rows and `.nunique()` on `Subject` and `Word`.

</details>

In [ ]:
check.equal(n_trials, 1659, hint="The first element of trials.shape is the number of rows.")
check.equal(n_participants, 21, hint="Count unique values in Subject with .nunique().")
check.equal(n_words, 79, hint="Count unique values in Word with .nunique().")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
n_trials = len(trials)
n_participants = trials["Subject"].nunique()
n_words = trials["Word"].nunique()
```

</details>

## 3 · Select columns

Create `analysis_columns` containing these columns in this order: `Subject`, `Word`,
`RT`, `NativeLanguage`, `Correct`, `Frequency`, `Length`, and `Class`.

In [ ]:
analysis_columns = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Select multiple columns with `trials[["first", "second", ...]]`.

</details>

In [ ]:
check.columns(
    analysis_columns,
    ["Subject", "Word", "RT", "NativeLanguage", "Correct", "Frequency", "Length", "Class"],
)
check.equal(len(analysis_columns), 1659, hint="Column selection should retain every trial.")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
analysis_columns = trials[[
    "Subject", "Word", "RT", "NativeLanguage",
    "Correct", "Frequency", "Length", "Class",
]]
```

</details>

## 4 · Accuracy and Boolean filtering

Count correct and incorrect responses. Then create `correct_trials` containing only
correct responses.

In [ ]:
n_correct = ...
n_incorrect = ...
correct_trials = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Compare `trials["Correct"]` with the strings `"correct"` and `"incorrect"`. Use the resulting Boolean Series to filter rows.

</details>

In [ ]:
check.equal(n_correct, 1594, hint="Use value_counts() or compare Correct with 'correct'.")
check.equal(n_incorrect, 65, hint="Use value_counts() or compare Correct with 'incorrect'.")
check.equal(set(correct_trials["Correct"]), {"correct"}, hint="Filter rows where Correct equals 'correct'.")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
n_correct = trials["Correct"].eq("correct").sum()
n_incorrect = trials["Correct"].eq("incorrect").sum()
correct_trials = trials.loc[trials["Correct"].eq("correct")]
```

There are 1,594 correct and 65 incorrect trials.

</details>

### Reflection · What changes when errors disappear?

Filtering to correct responses is common in reaction-time analyses. Which research
question can `correct_trials` answer, and which question can it no longer answer? Could
the filter affect conditions or participant groups unequally?

In [ ]:
reflection_filtering = """
After filtering, the data can answer ...
It can no longer answer ...
"""

<details class="notebook-reflection">
<summary><strong>Compare your reasoning</strong></summary>

The filtered data can describe response speed conditional on a correct answer. It
cannot describe accuracy or the complete speed–accuracy trade-off. If one group or
condition makes more errors, its remaining correct trials may be a selected subset, so
always inspect accuracy before interpreting reaction times.

</details>

## 5 · Convert reaction time

`RT` is the natural logarithm of reaction time in milliseconds. Add `RT_ms` to
`correct_trials` by applying `np.exp` to `RT`.

In [ ]:
correct_trials = correct_trials.copy()
correct_trials["RT_ms"] = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

The inverse of the natural logarithm is `np.exp`.

</details>

In [ ]:
check.equal(
    round(float(correct_trials["RT_ms"].median()), 1),
    571.0,
    hint="Use np.exp(correct_trials['RT']).",
)

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
correct_trials = correct_trials.copy()
correct_trials["RT_ms"] = np.exp(correct_trials["RT"])
```

The median correct-trial reaction time is approximately 571 ms.

</details>

## 6 · Summarise language groups

Calculate the median correct-trial reaction time for the two `NativeLanguage` groups.
Return a Series indexed by `NativeLanguage`.

In [ ]:
median_rt = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Group by `NativeLanguage`, select `RT_ms`, and call `.median()`.

</details>

In [ ]:
check.equal(round(float(median_rt.loc["English"]), 1), 541.5, hint="Group correct_trials by NativeLanguage and take the median of RT_ms.")
check.equal(round(float(median_rt.loc["Other"]), 1), 616.5, hint="Group correct_trials by NativeLanguage and take the median of RT_ms.")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
median_rt = (
    correct_trials
    .groupby("NativeLanguage")["RT_ms"]
    .median()
)
```

The medians are approximately 541.5 ms for the English group and 616.5 ms for the
Other group.

</details>

### Reflection · One number per group

What does the group median conceal? Name two distributions or levels of variation you
would inspect before describing one language group as “slower.”

In [ ]:
reflection_group_summary = """
The median conceals ...
I would inspect ...
"""

<details class="notebook-reflection">
<summary><strong>Compare your reasoning</strong></summary>

The two medians conceal variation between participants, variation between words, the
shape and tails of the reaction-time distributions, trial counts, and uncertainty.
Participant-level summaries and plots of the distributions are useful first checks;
an inferential analysis should respect repeated observations of participants and words.

</details>

## 7 · Word frequency and reaction time

Create one row per word with its frequency and mean correct reaction time. Plot word
frequency against mean reaction time.

Then answer:

1. What pattern is visible?
2. Why should the 1,659 trials not be treated as independent observations?
3. Which variables might confound a comparison between native-language groups?

In [ ]:
word_summary = ...

# Create a scatter plot of Frequency and mean_rt_ms.


# Interpretation:

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Group by `Word`, `Frequency`, and `Length` with `as_index=False`. Use named aggregation to create `mean_rt_ms`. Plot one point per row of the summary.

</details>

In [ ]:
check.equal(len(word_summary), 79, hint="Group by Word, Frequency, and Length.")
check.columns(word_summary, ["Word", "Frequency", "Length", "mean_rt_ms"])

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
word_summary = (
    correct_trials
    .groupby(["Word", "Frequency", "Length"], as_index=False)
    .agg(mean_rt_ms=("RT_ms", "mean"))
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(word_summary["Frequency"], word_summary["mean_rt_ms"], alpha=0.7)
ax.set(
    xlabel="Log word frequency",
    ylabel="Mean correct reaction time (ms)",
)
```

Higher-frequency words tend to have shorter reaction times. Trials are clustered within
participants and words, so the 1,659 rows are not independent. Word frequency, word
length, trial order, accuracy, and unequal participant composition could affect a simple
comparison between language groups.

</details>

### Reflection · From pattern to claim

Write one sentence that the scatter plot supports and one stronger sentence that it
does **not** support. What additional analysis or design information would you need for
the stronger claim?

In [ ]:
reflection_claim = """
Supported: ...
Not supported: ...
I would need ...
"""

<details class="notebook-reflection">
<summary><strong>Compare your reasoning</strong></summary>

The plot supports a descriptive statement such as “higher-frequency words tended to
have shorter mean reaction times in this dataset.” It does not by itself show that word
frequency caused faster responses. Word length, semantic class, participant effects,
and the sampling design would need attention in a model and interpretation.

</details>